In [0]:
"""
04_material_kpis.py

Streaming Material KPIs.

Computes supplier and material traceability KPIs.

Input:
    material_events

Output:
    material_kpis

Author:
Sumanth Vempalle

Version:
2.1.0
"""

import dlt

from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
    sum,
    when,
)


# ============================================================
# Material KPIs
# ============================================================

@dlt.table(
    name="material_kpis",
    comment="Material traceability and supplier performance KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dlt.expect_or_drop(
    "valid_supplier",
    "supplier IS NOT NULL",
)

@dlt.expect_or_drop(
    "valid_material_number",
    "material_number IS NOT NULL",
)

@dlt.expect(
    "positive_material_scans",
    "materials_scanned > 0",
)

def material_kpis():

    materials = (
         dlt.read_stream(
            "material_events"
    )

    .withWatermark(
        "event_timestamp",
        "10 minutes",
    )

)

    return (

        materials

        .groupBy(

            "plant_code",

            "supplier",

            "material_number",

            "product_code",

            "product_name",

            "family",

        )

        .agg(

            count("*").alias(
                "materials_scanned"
            ),

            sum(

                when(
                    col("scan_status") == "SUCCESS",
                    1,
                ).otherwise(0)

            ).alias(
                "successful_scans"
            ),

            sum(

                when(
                    col("scan_status") != "SUCCESS",
                    1,
                ).otherwise(0)

            ).alias(
                "failed_scans"
            ),

        )

        .withColumn(

            "scan_success_rate",

            (
                col("successful_scans")
                /
                col("materials_scanned")
            ) * 100

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )